# 2.7 - Clasificación de smartphones con Dropout

Repetimos `2_06_Regularizacion_Normalizacion_Clasificacion_Smartphones.ipynb`: predecimos una de las cuatro categorías de precio de un smartphone a partir de sus 20 características.

Conservamos la red **20–32–16–4**, la regularización **L2 (Ridge)** y **Batch Normalization**. Añadimos **Dropout con una tasa de 0,20 después de la ReLU de cada capa oculta**.

Seguimos el mismo flujo: carga y distribución de datos, partición en entrenamiento y test, transformación de entradas, arquitectura, pérdida y optimizador, entrenamiento, curvas, métricas y matrices de confusión.

Durante `fit`, Dropout anula aleatoriamente activaciones. Durante `predict` o al llamar al modelo con `training=False`, Dropout está desactivado. La tasa 0,20 es un punto de partida didáctico; no garantiza una mejora.

**Un único modelo, solo entrenamiento y test. Notebook preparado sin ejecutar.**

---
## 1. Carga de datos

Utilizamos el CSV de la asignatura. Las etiquetas se conservan como enteros; el target no se normaliza.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    log_loss, classification_report, ConfusionMatrixDisplay,
)
import tensorflow as tf
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense, BatchNormalization, Activation, Dropout
from tensorflow.keras.regularizers import L2
from tensorflow.keras.callbacks import Callback

SEMILLA = 42
tf.keras.utils.set_random_seed(SEMILLA)
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

rutas = [Path("datasets/Mobile_Price_Classification.csv"), Path("MUIA_RNMU/datasets/Mobile_Price_Classification.csv")]
ruta = next((p for p in rutas if p.is_file()), None)
if ruta is None:
    raise FileNotFoundError("Abre el notebook desde MUIA_RNMU o su carpeta superior.")
df = pd.read_csv(ruta)
target_col = "price_range"
CLASES = np.array([0, 1, 2, 3])
NOMBRES = ['Económico', 'Medio', 'Alto', 'Muy alto']
display(df.head(5))
print(f"{len(df):,} observaciones y {df.shape[1] - 1} variables de entrada.")
if df.isna().any().any():
    raise ValueError("Hay valores ausentes: revisa su tratamiento antes de continuar.")

### 1.1. Distribución del target

Contamos las observaciones y el porcentaje de cada clase y mostramos un gráfico de barras. Una accuracy elevada puede ocultar un mal resultado en una clase minoritaria: también examinaremos precision, recall y F1.

In [ ]:
conteos = df[target_col].value_counts().reindex(CLASES, fill_value=0)
distribucion = pd.DataFrame({
    "Clase": CLASES,
    "Significado": NOMBRES,
    "Observaciones": conteos.to_numpy(),
    "Porcentaje": 100 * conteos.to_numpy() / len(df),
})
display(distribucion)

print("Valora el desbalance junto con los errores de cada clase.")
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(NOMBRES, conteos.to_numpy(), color="#2F6BFF")
ax.set(title="Distribución del target", xlabel="Clase", ylabel="Observaciones")
ax.bar_label(ax.containers[0])
fig.tight_layout()
plt.show()

En el CSV proporcionado hay **500 smartphones por clase (25 % cada una)**: las cuatro clases están balanceadas. Comprobamos esta distribución al cargarlo en lugar de suponerla.

---
## 2. Partición en entrenamiento y test

Reservamos el 20 % para test. `stratify=y` conserva aproximadamente la proporción de clases de la muestra. Test no actualiza los pesos y no se utiliza para elegir épocas, arquitectura, umbral ni hiperparámetros.

In [ ]:
X = df.drop(columns=target_col).copy()
y = df[target_col].astype("int32")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)
y_train_array = y_train.to_numpy()
y_test_array = y_test.to_numpy()
print(f"Entrenamiento: {len(X_train)} | Test: {len(X_test)}")
display(pd.DataFrame({
    "Train (%)": y_train.value_counts(normalize=True).reindex(CLASES) * 100,
    "Test (%)": y_test.value_counts(normalize=True).reindex(CLASES) * 100,
}))

---
## 3. Transformación y normalización

Las variables numéricas se escalan con **MinMaxScaler**. Las categóricas se representan mediante **one-hot**, sin inventar un orden. Los codificadores y el escalador se ajustan solo con entrenamiento y se reutilizan en test.

Las características del CSV de smartphones ya son numéricas, incluidas las variables binarias 0/1. No hacemos one-hot del target: las clases 0–3 se usarán directamente con la pérdida sparse.

In [ ]:
numericas = X_train.select_dtypes(include="number").columns.tolist()
categoricas = X_train.select_dtypes(exclude="number").columns.tolist()
preprocesador = ColumnTransformer([
    ("numericas", MinMaxScaler(), numericas),
    ("categoricas", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categoricas),
], sparse_threshold=0)

X_train_n = preprocesador.fit_transform(X_train).astype("float32")
X_test_n = preprocesador.transform(X_test).astype("float32")
print("Dimensión de entrada tras transformar:", X_train_n.shape[1])
display(pd.DataFrame(X_train_n[:5], columns=preprocesador.get_feature_names_out()))

---
## 4. Arquitectura de la red neuronal

Cada capa oculta sigue **Dense → Batch Normalization → ReLU → Dropout**. Las capas Dense mantienen la penalización **L2**, y Dropout utiliza una tasa de **0,20**.

La salida conserva cuatro neuronas **softmax**. La clase predicha es la de mayor probabilidad (`argmax`).

In [ ]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEMILLA)

LAMBDA_L2 = 0.001
DROPOUT_RATE = 0.20

model = Sequential([
    Input(shape=(X_train_n.shape[1],)),

    # Primera capa oculta: regularización L2 y Batch Normalization.
    Dense(32, use_bias=False, kernel_regularizer=L2(LAMBDA_L2)),
    BatchNormalization(),
    Activation("relu"),
    Dropout(DROPOUT_RATE),

    # Segunda capa oculta: regularización L2 y Batch Normalization.
    Dense(16, use_bias=False, kernel_regularizer=L2(LAMBDA_L2)),
    BatchNormalization(),
    Activation("relu"),
    Dropout(DROPOUT_RATE),

    Dense(4, activation="softmax"),
])
model.summary()

---
## 5. Pérdida, optimizador y métrica de entrenamiento

Elegimos **sparse categorical crossentropy** porque el target contiene índices enteros de clase, no vectores one-hot. Si el target fuera one-hot, correspondería `categorical_crossentropy`.

Usamos **Adam**, con tasa de aprendizaje 0,001. La única métrica de `compile` es **accuracy**. Precision, recall y F1 se calcularán después del entrenamiento.

Keras añade automáticamente la penalización L2 al objetivo que minimiza durante `fit`.

In [ ]:
EPOCHS = 50
BATCH_SIZE = 32
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

---
## 6. Entrenamiento: mostramos solo accuracy

Al acabar cada época, guardamos las probabilidades de entrenamiento y test con los mismos pesos. El callback **solo muestra accuracy**. Una vez finalizado `fit`, utilizaremos las probabilidades guardadas para calcular todas las curvas, sin reentrenar ni modificar los pesos.

Las predicciones se guardan en memoria para este ejemplo pequeño. Las consultas a test son solo didácticas: si decidimos cambios mirando esas curvas, test deja de ser una evaluación final independiente. No usamos `validation_data` ni un tercer conjunto.

`fit` activa Dropout automáticamente. El seguimiento utiliza `training=False`: las curvas se calculan con Dropout desactivado y sin actualizar las estadísticas de Batch Normalization con test.

In [ ]:
def obtener_clases(probabilidades):
    return np.argmax(probabilidades, axis=1)


class GuardarEvolucion(Callback):
    def on_train_begin(self, logs=None):
        self.probabilidades_por_epoch = []

    def on_epoch_end(self, epoch, logs=None):
        p_train = self.model(X_train_n, training=False).numpy()
        p_test = self.model(X_test_n, training=False).numpy()
        self.probabilidades_por_epoch.append((p_train, p_test))
        acc_train = accuracy_score(y_train_array, obtener_clases(p_train))
        acc_test = accuracy_score(y_test_array, obtener_clases(p_test))
        print(f"Epoch {epoch + 1:02d}/{self.params['epochs']} | "
              f"accuracy train: {acc_train:.4f} | accuracy test: {acc_test:.4f}")


seguimiento = GuardarEvolucion()
history = model.fit(
    X_train_n, y_train_array,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=[seguimiento], shuffle=True, verbose=0,
)

---
## 7. Cálculo de las métricas después de entrenar

* **Accuracy:** proporción global de etiquetas correctas.
* **Precision:** de los positivos predichos, proporción que realmente es positiva.
* **Recall:** de los positivos reales, proporción detectada.
* **F1:** media armónica de precision y recall.

En multiclase calculamos precision, recall y F1 **macro**: cada clase se considera positiva frente al resto y se promedian los resultados dando el mismo peso a cada clase. F1 macro es el promedio de los F1 individuales, no la media armónica de precision macro y recall macro.

Usamos `zero_division=0`: si no hay predicciones positivas para una clase, una división no definida se representa como 0. La pérdida se calcula con las probabilidades; las otras métricas, con las clases predichas.

In [ ]:
def calcular_metricas(reales, probabilidades):
    predichas = obtener_clases(probabilidades)
    return {
        "loss": log_loss(reales, probabilidades, labels=CLASES),
        "accuracy": accuracy_score(reales, predichas),
        "precision": precision_score(reales, predichas, average="macro", labels=CLASES, zero_division=0),
        "recall": recall_score(reales, predichas, average="macro", labels=CLASES, zero_division=0),
        "f1": f1_score(reales, predichas, average="macro", labels=CLASES, zero_division=0),
    }


registros = []
for epoch, (p_train, p_test) in enumerate(seguimiento.probabilidades_por_epoch, start=1):
    fila = {"epoch": epoch}
    for nombre, reales, probabilidades in [
        ("train", y_train_array, p_train), ("test", y_test_array, p_test)
    ]:
        for metrica, valor in calcular_metricas(reales, probabilidades).items():
            fila[f"{metrica}_{nombre}"] = valor
    registros.append(fila)
evolucion = pd.DataFrame(registros).set_index("epoch")
display(evolucion.tail(5))

---
## 8. Curvas de entrenamiento y test

Primero representamos **pérdida y accuracy** en una fila de dos columnas. Después, **precision, recall y F1** en otra fila de tres columnas. Se muestran todas las épocas. Azul indica entrenamiento y rojo test.

Estos valores se calculan al terminar cada época, con los mismos pesos para los dos conjuntos. Pueden diferir de los promedios internos por lote que registra `history`, porque durante una época los pesos van cambiando.

La curva de pérdida muestra **crossentropy sin penalización L2**, calculada con las probabilidades. El valor `history.history["loss"]` incluye la penalización L2 durante el entrenamiento.

In [ ]:
def dibujar_metrica(ax, metrica, titulo, es_proporcion=True):
    ax.plot(evolucion.index, evolucion[f"{metrica}_train"],
            color="#2F6BFF", label="Entrenamiento")
    ax.plot(evolucion.index, evolucion[f"{metrica}_test"],
            color="#FF3B30", label="Test")
    ax.set(title=titulo, xlabel="Epoch", ylabel=titulo)
    if es_proporcion:
        ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.25)
    ax.legend()


fig, axes = plt.subplots(1, 2, figsize=(13, 4))
dibujar_metrica(axes[0], "loss", "Pérdida: crossentropy", es_proporcion=False)
dibujar_metrica(axes[1], "accuracy", "Accuracy")
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, metrica, titulo in zip(axes, ["precision", "recall", "f1"], ["Precision", "Recall", "F1"]):
    dibujar_metrica(ax, metrica, titulo)
fig.tight_layout()
plt.show()

---
## 9. Evaluación final y resultados por clase

Calculamos las métricas finales sobre entrenamiento y test y mostramos un informe por clase. Como referencia sencilla, calculamos la accuracy de predecir siempre la clase mayoritaria **del entrenamiento**.

Consulta `BonusTrack_03_Evaluacion_Modelos_Clasificacion.ipynb` para ver los cálculos de las métricas con ejemplos pequeños y la interpretación de los promedios.

In [ ]:
p_train_final = model.predict(X_train_n, verbose=0)
p_test_final = model.predict(X_test_n, verbose=0)
pred_train = obtener_clases(p_train_final)
pred_test = obtener_clases(p_test_final)

resultados = pd.DataFrame({
    "Entrenamiento": calcular_metricas(y_train_array, p_train_final),
    "Test": calcular_metricas(y_test_array, p_test_final),
}).T
display(resultados)
clase_mayoritaria = y_train.value_counts().idxmax()
pred_base = np.full(len(y_test_array), clase_mayoritaria)
print(f"Accuracy base en test (siempre clase {clase_mayoritaria}): "
      f"{accuracy_score(y_test_array, pred_base):.4f}")

for nombre, reales, predichas in [
    ("Entrenamiento", y_train_array, pred_train),
    ("Test", y_test_array, pred_test),
]:
    print(f"\n{nombre}")
    print(classification_report(reales, predichas, labels=CLASES,
                                target_names=NOMBRES, zero_division=0, digits=4))

### 9.1. Comparación visual de las métricas finales

El panel de **dos filas y dos columnas** compara entrenamiento (azul) y test (naranja) para el único modelo de red neuronal. Cada pareja de barras representa accuracy, precision, recall o F1 al finalizar el entrenamiento. Todos los ejes tienen la misma escala y las barras muestran el valor numérico.

Precision, recall y F1 son promedios macro, con el mismo peso para cada clase.

In [ ]:
metricas_panel = ["accuracy", "precision", "recall", "f1"]
titulos_panel = ["ACCURACY", "PRECISION", "RECALL", "F1"]
comparacion_metricas = resultados.loc[["Entrenamiento", "Test"], metricas_panel]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
ancho = 0.32
for ax, metrica, titulo in zip(axes.flat, metricas_panel, titulos_panel):
    valor_train = float(comparacion_metricas.loc["Entrenamiento", metrica])
    valor_test = float(comparacion_metricas.loc["Test", metrica])
    barras_train = ax.bar(-ancho / 2, valor_train, width=ancho, label="Entrenamiento")
    barras_test = ax.bar(ancho / 2, valor_test, width=ancho, label="Test")
    ax.bar_label(barras_train, fmt="%.4f", padding=3)
    ax.bar_label(barras_test, fmt="%.4f", padding=3)
    ax.set(title=titulo, ylabel="Valor", ylim=(0, 1.08), xlim=(-0.8, 0.8))
    ax.set_xticks([0], ["Red neuronal"])
    ax.set_yticks(np.arange(0, 1.01, 0.2))
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(loc="upper right")

fig.suptitle("Comparación de métricas del modelo final", fontsize=14)
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

---
## 10. Matrices de confusión

Las **filas representan clases reales** y las **columnas clases predichas**. La diagonal contiene los aciertos. Usamos el mismo orden de clases en los dos conjuntos y mostramos **recuentos**, no porcentajes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, reales, predichas, titulo in [
    (axes[0], y_train_array, pred_train, "Entrenamiento"),
    (axes[1], y_test_array, pred_test, "Test"),
]:
    ConfusionMatrixDisplay.from_predictions(
        reales, predichas, labels=CLASES, display_labels=NOMBRES,
        values_format="d", cmap="Blues", colorbar=False, ax=ax,
    )
    ax.set(title=f"Matriz de confusión: {titulo}",
           xlabel="Clase predicha", ylabel="Clase real")
fig.tight_layout()
plt.show()